In [1]:
# import pyearthtools.pipeline as petpipe

# Lists and Dictionaries Required For Sustom Pipeline Steps

In [ ]:
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.,
        "winddirs": -999.
    }

flagged_labels = [
        'temperatures', 'dewpoints', 'slp',
        'stnlp', 'windspeeds', 'winddirs', 
        'total_cloud_cover', 'low_cloud_cover', 'mid_cloud_cover', 
        'high_cloud_cover', 'precip1_depth', 'precip2_depth', 
        'precip3_depth', 'precip6_depth', 'precip9_depth',
        'precip12_depth', 'precip15_depth', 'precip18_depth', 
        'precip24_depth'
    ]

# Creation of Custom PipelineSteps 
This is a good demonstration of how custom pipeline steps can be created and add to pipelines, without the need to modify source code!

In [ ]:
# Custom operation to remove redundent coordinates
class SqueezeStationCoordinates(petpipe.Operation):
    def __init__(self, coords=("latitude", "longitude", "elevation")):
        super().__init__()
        self.coords = coords
    def apply_func(self, ds):
        for coord in self.coords:
            if coord in ds.coords and len(ds[coord].shape) > 1:
                ds[coord] = ds[coord].squeeze()
        return ds
    
    # Undo function added otherwise pyearthtools will complain
    def undo_func(self, ds):
        # No undo operation needed for this operation
        return ds

In [ ]:
class TrainTestSplit(petpipe.Operation):
    """
    Split dataset into train and test sets along a specified dimension.
    """
    def __init__(self, test_size=0.2, random_state=42, dim='time'):
        super().__init__()
        self.test_size = test_size
        self.random_state = random_state
        self.dim = dim

    def apply_func(self, ds):
        import numpy as np
        from sklearn.model_selection import train_test_split

        idx = np.arange(ds.sizes[self.dim])
        train_idx, test_idx = train_test_split(
            idx, test_size=self.test_size, random_state=self.random_state
        )
        ds_train = ds.isel({self.dim: train_idx})
        ds_test = ds.isel({self.dim: test_idx})
        return ds_train, ds_test

    def undo_func(self, ds):
        return ds

In [ ]:
# Custom operation split data into feature and target variables
class FeatureTargetSplit(petpipe.Operation):
    """
    Split dataset into features (X) and target (y) for ML.
    """
    def __init__(self, target_vars, drop_target_from_x=True):
        super().__init__()
        self.target_vars = [target_vars] if isinstance(target_vars, str) else list(target_vars)
        self.drop_target_from_x = drop_target_from_x

    def apply_func(self, ds):
        # X: all variables except target(s)
        if self.drop_target_from_x:
            feature_vars = [v for v in ds.data_vars if v not in self.target_vars]
        else:
            feature_vars = list(ds.data_vars)
        X = ds[feature_vars]
        # y: just the target(s)
        y = ds[self.target_vars]
        return (X, y)
    
    # Undo function added otherwise pyearthtools will complain
    def undo_func(self, ds):
        return ds

In [ ]:
class MedianImputePerStation(petpipe.Operation):
    """
    Impute missing values in X (numpy array) with the median per station for each feature.
    Assumes input shape: (features, stations, time)
    """
    def __init__(self):
        super().__init__()

    def apply_func(self, x):
        x_imputed = x.copy()
        for i in range(x.shape[0]):  # for each feature
            for j in range(x.shape[1]):  # for each station
                arr = x[i, j, :]
                median = np.nanmedian(arr)
                arr_imputed = np.where(np.isnan(arr), median, arr)
                x_imputed[i, j, :] = arr_imputed
        return x_imputed

    def undo_func(self, x):
        # No undo operation needed
        return x
    

In [ ]:
class SelectAndFlattenY(petpipe.Operation):
    """
    Selects a specific test from y and flattens it.
    Assumes y shape: (1, stations, time, n_tests)
    """
    def __init__(self, test_number):
        super().__init__()
        self.test_number = test_number

    def apply_func(self, y):
        y_selected = y[0, :, :, self.test_number]
        y_flat = y_selected.reshape(-1)
        return y_flat

    def undo_func(self, y):
        return y  # No undo


In [ ]:

class TransposeAndFlattenX(petpipe.Operation):
    """
    Transposes X to (stations, time, features) and flattens to 2D.
    Assumes X shape: (features, stations, time)
    """
    def apply_func(self, x):
        x_transposed = np.transpose(x, (1, 2, 0))  # (stations, time, features)
        X_flat = x_transposed.reshape(-1, x.shape[0])  # (stations*time, features)
        return X_flat

    def undo_func(self, x):
        return x  # No undo

In [ ]:
class FilterNaNSamples(petpipe.Operation):
    """
    Filters out samples where y is NaN.
    Assumes input is a tuple (X, y), both as numpy arrays with matching first dimension.
    """
    def apply_func(self, tup):
        X, y = tup
        mask = ~np.isnan(y)
        X_final = X[mask]
        y_final = y[mask]
        return X_final, y_final

    def undo_func(self, tup):
        return tup  # No undo

## Test SetMissingToNaN

In [ ]:
import xarray as xr
from pyearthtools.data.transforms.values import SetMissingToNaN
import numpy as np

def test_set_missing_to_nan():
    data = xr.Dataset({
        "total_cloud_cover": ("time", [0, -999, 50]),
        "low_cloud_cover": ("time", [10, -999, 20]),
    })

    varname_val_map = {
        "total_cloud_cover": -999.0,
        "low_cloud_cover": -999.0,
    }

    transform = SetMissingToNaN(varname_val_map)
    transformed_data = transform.apply(data)

    if np.isnan(transformed_data["total_cloud_cover"].data[1]):
        print("Total cloud cover at index 1 is NaN")
    else:
        print("Total cloud cover at index 1 is not NaN")
    
    if np.isnan(transformed_data["low_cloud_cover"].data[1]):
        print("Low cloud cover at index 1 is NaN")
    
    if transformed_data["total_cloud_cover"].data[2] == 50:
        print("Total cloud cover at index 2 is 50")

In [ ]:
# test_set_missing_to_nan() # Uncomment to run the test

## Test AddFlaggedObs

In [ ]:
import xarray as xr
import numpy as np
from pyearthtools.data.transforms.values import AddFlaggedObs

def test_add_flagged_obs():
    # Mock dataset with flagged observations
    data = xr.Dataset(
        {
            "temperatures": (("time",), [np.nan, 15.0, np.nan]),
            "dewpoints": (("time",), [np.nan, 10.0, np.nan]),
            "flagged_obs": (
                ("time", "flagged"),
                [
                    [20.0, np.nan],  # Flagged data for time=0
                    [np.nan, np.nan],  # No flagged data for time=1
                    [25.0, 12.0],  # Flagged data for time=2
                ],
            ),
        },
        coords={
            "time": [0, 1, 2],
            "flagged": [0, 1],  # Indices for flagged variables
        },
    )

    # Add flagged_value attribute to variables
    data["temperatures"].attrs["flagged_value"] = -999.0
    data["dewpoints"].attrs["flagged_value"] = -999.0

    # Define flagged labels corresponding to the flagged dimension
    flagged_labels = ["temperatures", "dewpoints"]

    # Apply the AddFlaggedObs transform
    transform = AddFlaggedObs(flagged_labels)
    transformed_data = transform.apply(data)

    # Assert that flagged data has been restored
    assert np.allclose(
        transformed_data["temperatures"].data, [20.0, 15.0, 25.0], equal_nan=True
    )
    assert np.allclose(
        transformed_data["dewpoints"].data, [np.nan, 10.0, 12.0], equal_nan=True
    )

    print("Test passed: Flagged observations were correctly restored.")



In [ ]:
# test_add_flagged_obs() # Uncomment to run the test